> **The 45-second mystery:** Your fine-tuning loop from `04-llm` takes 45 seconds per epoch on your machine. A colleague with "identical" hardware runs the same code in 8 seconds. Before optimizing anything, you need to *measure*: where does the time actually go? Is the bottleneck loading data? Running the forward pass? The backward pass? The optimizer step? Guessing wrong wastes days of engineering.
>
> This chapter teaches the profiling habit: measure first, optimize second. Every optimization technique in Chapters 4–8 makes sense only after you've measured which operation is the actual bottleneck on your specific hardware.

# PyTorch Profiling: Finding the Real Bottleneck

| Part | Tool | Question answered |
|------|------|------------------|
| 1 | `torch.profiler` | Which operators take the most time in a forward pass? |
| 2 | `autograd.profiler` | How much does profiling itself cost? |
| 3 | Compute vs. memory bound | Is our attention compute-bound or memory-bound? |
| 4 | `torch.compile` | When does graph compilation actually help? |
| 5 | Custom profiling regions | How do I isolate one section of a training loop? |
| 6 | End-to-end step timing | Where does the 45 seconds actually go? |

---

## Prerequisite Bridge — From Ch1 GPU Hardware

| Foundation | Role in this notebook |
|---|---|
| Arithmetic intensity (FLOP/byte) | Determines whether profiling will show compute or bandwidth as the limit |
| Roofline model | Interprets profiler output: is the bottleneck hitting the compute ceiling or the bandwidth ceiling? |
| Memory coalescing | Explains why some operators are slower than expected in profiles |

> **If you haven't read `learning/ai-infrastructure/01-gpu-hardware/gpu-hardware-foundations.ipynb`** the compute-bound vs. memory-bound distinction used in Part 3 will be unfamiliar.

In [ ]:
import subprocess, sys
for pkg in ['torch', 'numpy', 'matplotlib', 'transformers']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()
torch.manual_seed(42)

print(f"Device: {DEVICE}")
if HAS_GPU:
    print(f"GPU: {torch.cuda.get_device_properties(0).name}")
else:
    print("No GPU — profiling runs on CPU. Key patterns and ratios are identical.")
    print("Absolute times will be slower than GPU; relative proportions are instructive.")

print()
# ── Running example: small GPT-2-style Transformer ───────────────────────────
B, S, D = 4, 128, 256   # batch=4, seq=128, d_model=256
N_HEADS  = 8
N_LAYERS = 6
VOCAB    = 1000
print(f"Running example: Transformer (B={B}, S={S}, D={D}, {N_HEADS} heads, {N_LAYERS} layers)")

In [ ]:
# ── Build a small Transformer for profiling ───────────────────────────────────
class SmallTransformer(nn.Module):
    """Small GPT-like model for profiling demonstrations."""
    def __init__(self, vocab=VOCAB, d=D, n_heads=N_HEADS, n_layers=N_LAYERS, seq=S):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        encoder_layer = nn.TransformerEncoderLayer(d, n_heads, dim_feedforward=d*4,
                                                    batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

model = SmallTransformer().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SmallTransformer: {n_params/1e6:.1f}M parameters")

# Sample batch
x_batch = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
labels  = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Warm up
for _ in range(3):
    out = model(x_batch)
    loss = criterion(out.view(-1, VOCAB), labels.view(-1))
    loss.backward()
    optimizer.zero_grad()
if HAS_GPU: torch.cuda.synchronize()
print("Model warmed up and ready for profiling.")

---

## Part 1 — `torch.profiler`: The Full Operator Timeline

`torch.profiler` captures every operator call during a context window, with CPU and (optionally) CUDA timing. It's the most comprehensive tool but has overhead (~5–10% slowdown).

#### 🔮 Predict first

For one forward + backward pass through our SmallTransformer, rank the four phases by wall time:

1. **(a) data_prep > forward > backward > optimizer** — moving data dominates
2. **(b) backward > forward > optimizer > data_prep** — backward is most expensive (it computes all gradients)
3. **(c) optimizer > backward > forward > data_prep** — AdamW update is most expensive

Which order do you expect? The answer depends on your hardware and model size — that's exactly the lesson.

![Profiler timeline: data_prep (teal) + forward (amber) + backward (coral, widest) + optimizer_step (green) annotated with operator names](images/profiler-timeline-annotated.png)

In [ ]:
# ── Part 1: torch.profiler ────────────────────────────────────────────────────
from torch.profiler import profile, record_function, ProfilerActivity

# Profile one forward+backward step
activities = [ProfilerActivity.CPU]
if HAS_GPU: activities.append(ProfilerActivity.CUDA)

optimizer.zero_grad()
with profile(activities=activities, record_shapes=True, with_stack=False) as prof:
    with record_function("data_prep"):
        x_b = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
        y_b = torch.randint(0, VOCAB, (B, S)).to(DEVICE)

    with record_function("forward"):
        logits = model(x_b)
        loss = criterion(logits.view(-1, VOCAB), y_b.view(-1))

    with record_function("backward"):
        loss.backward()

    with record_function("optimizer_step"):
        optimizer.step()
        optimizer.zero_grad()

if HAS_GPU: torch.cuda.synchronize()

# Extract timings
key_averages = prof.key_averages()
print("Top 10 operators by CPU time:")
print(key_averages.table(sort_by="cpu_time_total", row_limit=10))

#### How to read this profiler table

| Column | What it means | Action |
|---|---|---|
| `self_cpu_time_total` | Time *inside* this op, excluding child ops | **Sort by this** to find the actual hot op |
| `cpu_time_total` | Time including all child ops in the subtree | High here + low `self` = caller, not the bottleneck |
| `cuda_time_total` | Matching GPU kernel time | Large gap vs CPU = async overlap (usually good) |
| `count` | How many times this op ran | High count + low self = loop overhead, not kernel cost |

**Rule of thumb:** Sort by `self_cpu_time_total` to find what's actually slow. `cpu_time_total` shows where the call stack is deepest — useful for callers, not causes.

In [ ]:
# ── Part 1b: Phase-level timing ───────────────────────────────────────────────
phases = ['data_prep', 'forward', 'backward', 'optimizer_step']
phase_times = {}

for phase in phases:
    events = [e for e in key_averages if e.key == phase]
    if events:
        t = events[0].cpu_time_total / 1000  # μs → ms
        phase_times[phase] = t
    else:
        phase_times[phase] = 0.0

total_t = sum(phase_times.values()) or 1.0

print("Phase timing breakdown:")
for phase, t in sorted(phase_times.items(), key=lambda x: -x[1]):
    pct = t / total_t * 100
    bar = "\u2588" * int(pct / 2)
    print(f"  {phase:20s}: {t:7.2f} ms  ({pct:5.1f}%)  {bar}")

# Determine actual order
sorted_phases = sorted(phase_times.items(), key=lambda x: -x[1])
order = [p for p, _ in sorted_phases]
print()
print(f"Actual order (slowest first): {' > '.join(order)}")
print()
print("Prediction check:")
print("  Answer depends on hardware — the point is that the profiler told us, not our guess.")
print(f"  {'backward' if order[0] == 'backward' else order[0]} is the dominant phase on this machine.")

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
color_map = {'data_prep': 'steelblue', 'forward': '#f0a500', 'backward': 'coral', 'optimizer_step': 'mediumseagreen'}
color_list = [color_map.get(p, 'gray') for p in phase_times.keys()]
bars = ax.bar(phase_times.keys(), phase_times.values(), color=color_list)
ax.set_ylabel('Time (ms)'); ax.set_title('Single step phase timing')
for bar, (p, t) in zip(bars, phase_times.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{t:.1f}ms',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

#### What just happened — and what's missing

`torch.profiler` identified the most expensive phase and the top operators. The backward pass is typically 2–3× the forward pass time (it computes gradients for every parameter). On CPU, data movement to device shows up in data_prep; on GPU it's often negligible once data is already on device.

**Missing piece:** `torch.profiler` captures everything — including its own overhead. How much slower is profiled code vs. uninstrumented code? That's Part 2.

---

## Part 2 — `autograd.profiler`: Targeted Profiling

`torch.autograd.profiler.profile` is lighter-weight than `torch.profiler` — useful when you want to profile just one specific cell without the full trace overhead.

**Two-sided health check (Section 14.2):**
- **Too much profiling:** full `torch.profiler` on every training step slows the run by 5–15%
- **Too little:** no profiling at all means never knowing where the bottleneck is

In [ ]:
# ── Part 2: Profiling overhead measurement (two-sided health check) ───────────
def time_step(n_runs=20):
    """Time one forward+backward step without profiling."""
    if HAS_GPU: torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        optimizer.zero_grad()
        if HAS_GPU: torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model(x_batch); loss = criterion(out.view(-1, VOCAB), labels.view(-1))
        loss.backward(); optimizer.step()
        if HAS_GPU: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000  # ms

def time_step_profiled(n_runs=5):
    """Time one forward+backward step with full profiler enabled."""
    times = []
    for _ in range(n_runs):
        optimizer.zero_grad()
        t0 = time.perf_counter()
        with profile(activities=activities) as p:
            out = model(x_batch); loss = criterion(out.view(-1, VOCAB), labels.view(-1))
            loss.backward(); optimizer.step()
        if HAS_GPU: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000

baseline_ms     = time_step()
profiled_ms     = time_step_profiled()
overhead_pct    = (profiled_ms - baseline_ms) / baseline_ms * 100

print(f"Profiling overhead measurement:")
print(f"  Uninstrumented step:  {baseline_ms:.2f} ms")
print(f"  With torch.profiler:  {profiled_ms:.2f} ms")
print(f"  Overhead:             {overhead_pct:.1f}%")
print()
if overhead_pct < 20:
    print("\u2713 Health check (too-tight): profiling overhead is acceptable for occasional profiling runs")
    print("  \u2192 Use torch.profiler on 1-2 steps, not every step")
else:
    print(f"! Overhead is {overhead_pct:.0f}% — only profile during debugging, not production training")
print()
print("\u2713 Health check (too-loose): uninstrumented baseline gives reference timing")
print("  \u2192 Always run uninstrumented first to establish your baseline")

**Why does the profiler add overhead?**

The profiler intercepts every PyTorch operator call at the dispatcher level. Each interception records a timestamp, allocates a string (op name), and writes to the event buffer. The cost scales with **op-call count**, not compute time — so a model with many small ops (e.g., 1000 element-wise operations) sees more overhead than a model with few large matmuls (e.g., 5 large linear layers). This is why `with_stack=False` is faster than `with_stack=True`.

---

## Part 3 — Compute-Bound vs. Memory-Bound: Which Limit Are We Hitting?

Two operations look similar in source code but perform very differently:
- **Matrix multiply** (Q @ K.T): many FLOPs per byte of memory → compute-bound
- **Softmax** (along S axis): few FLOPs per byte → memory-bound

A profiler showing high kernel time for softmax suggests a memory-bandwidth bottleneck (consistent with Ch1's roofline model: attention softmax has low arithmetic intensity).

![Roofline diagram: matmul (coral) sits near the compute ceiling; attention softmax (amber) sits far left in the memory-bound region](images/compute-vs-memory-bound.png)

In [ ]:
# ── Part 3: Compare attention matmul vs. softmax timing ──────────────────────
B_bench, S_bench, D_bench = 8, 512, 64
Q = torch.randn(B_bench, S_bench, D_bench).to(DEVICE)
K = torch.randn(B_bench, S_bench, D_bench).to(DEVICE)

def bench_op(op_fn, n=30, sync=HAS_GPU):
    if sync: torch.cuda.synchronize()
    times = []
    for _ in range(n):
        if sync: torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = op_fn()
        if sync: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000, result

# Op 1: matmul Q @ K^T → high arithmetic intensity (compute-bound)
t_matmul, scores = bench_op(lambda: torch.matmul(Q, K.transpose(-2, -1)))

# Op 2: softmax over S dim → low arithmetic intensity (memory-bound)
t_softmax, _ = bench_op(lambda: torch.softmax(scores, dim=-1))

# Op 3: both together (standard attention)
t_full, _ = bench_op(lambda: torch.softmax(torch.matmul(Q, K.transpose(-2,-1)) / (D_bench**0.5), dim=-1))

matmul_flops = 2 * B_bench * S_bench * S_bench * D_bench / 1e9  # GFLOP
softmax_bytes = B_bench * S_bench * S_bench * 4 * 3 / 1e9  # 3x for read/compute/write

print(f"Attention operations at (B={B_bench}, S={S_bench}, D={D_bench}):")
print(f"  Q @ K^T (matmul):  {t_matmul:.3f} ms  |  {matmul_flops/t_matmul*1000:.1f} GFLOPS  (likely compute-bound)")
print(f"  softmax(scores):   {t_softmax:.3f} ms  |  {softmax_bytes/t_softmax*1000:.1f} GB/s effective BW")
print(f"  Full attention:    {t_full:.3f} ms")
print()
# Reference peaks for interpretation
print("\n→ Hardware reference peaks (for context):")
print("   A100 SXM: ~312 TFLOPS (fp16) peak compute, 2.0 TB/s peak bandwidth")
print("   RTX 4090: ~330 TFLOPS (fp16) peak compute, 1.0 TB/s peak bandwidth")
print("   A10G:     ~125 TFLOPS (fp16) peak compute, 0.6 TB/s peak bandwidth")
print(f"\n→ Your measured GFLOPS / 312,000 = fraction of A100 compute used")
print("→ If < 10%: memory-bound (fix memory access pattern)")
print("→ If > 50%: compute-bound (batch larger, use tensor cores)")
if t_matmul < t_softmax:
    print("\u2192 On this machine: softmax takes LONGER than the matmul despite fewer FLOPs")
    print("  This is the memory-bandwidth bottleneck: softmax reads/writes a (S\u00d7S) matrix repeatedly")
    print("  FlashAttention (Ch4) solves exactly this: keeps the S\u00d7S intermediate in SRAM")
else:
    print("\u2192 On this machine: matmul dominates (likely a small GPU or CPU)")
    print("  On data-center GPUs (A100, H100): softmax often becomes the bottleneck at S=2048+")

#### What just happened — and what's missing

The profiler revealed that softmax over the attention matrix can be slower than the matmul on memory-bandwidth-limited hardware. The S×S matrix must be read and written multiple times (numerics stabilization, normalization) — each pass is a full HBM roundtrip.

**Missing piece:** We identified the bottleneck, but haven't fixed it. Can we fuse the matmul and softmax so the S×S matrix never leaves SRAM? Yes — that's FlashAttention (Ch4). But first, how much does `torch.compile` help without changing the algorithm?

---

## Part 4 — `torch.compile`: Graph Compilation

`torch.compile` (PyTorch 2.0) compiles a model using TorchDynamo (graph capture) and inductor (backend optimization). It fuses operators, removes Python overhead, and generates optimized kernels — without changing your code.

**When it helps:** Models with many small ops (lots of element-wise operations, activations).
**When it doesn't:** Models that are already memory-bandwidth-limited; the bottleneck is hardware, not Python overhead.

In [ ]:
# ── Part 4: torch.compile speedup measurement ────────────────────────────────
model_eager = SmallTransformer().to(DEVICE)
model_eager.load_state_dict(model.state_dict())  # same weights

try:
    model_compiled = torch.compile(SmallTransformer().to(DEVICE), mode='default')
    model_compiled.load_state_dict(model.state_dict())
    COMPILE_AVAILABLE = True
    print("torch.compile available")
except Exception as e:
    print(f"torch.compile not available: {e}")
    COMPILE_AVAILABLE = False

def time_inference(m, n_runs=30, warmup=5):
    m.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = m(x_batch)
        if HAS_GPU: torch.cuda.synchronize()
        times = []
        for _ in range(n_runs):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = m(x_batch)
            if HAS_GPU: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return np.median(times) * 1000

t_eager    = time_inference(model_eager)
print(f"\nEager mode (inference): {t_eager:.2f} ms")

if COMPILE_AVAILABLE:
    # First call triggers compilation (warm-up cost)
    print("Compiling model (first call triggers compilation)...")
    t0_compile = time.perf_counter()
    model_compiled.eval()
    with torch.no_grad(): _ = model_compiled(x_batch)
    compile_overhead_s = time.perf_counter() - t0_compile

    t_compiled = time_inference(model_compiled)
    speedup = t_eager / t_compiled

    print(f"Compiled mode (inference): {t_compiled:.2f} ms  ({speedup:.1f}\u00d7 speedup after warm-up)")
    print(f"Compilation overhead: {compile_overhead_s:.1f}s (one-time cost)")
    print()
    if speedup > 1.1:
        print("\u2192 torch.compile improved throughput on this architecture")
    else:
        print("\u2192 Speedup is modest \u2014 likely memory-bandwidth-limited (compile helps compute, not bandwidth)")
    print(f"  Break-even at {compile_overhead_s / max(t_eager - t_compiled, 0.001) * 1000:.0f} forward passes")
else:
    print("\n\u2192 Use torch.compile when: many small ops, compute-bound, model runs >1000 times")
    print("  Skip when: memory-bandwidth-limited, model architecture changes frequently")

---

## Part 5 — Custom Profiling Regions with `record_function`

For a long training loop, you often want to profile just one section without capturing everything. `torch.profiler.record_function` creates a named region visible in the Chrome trace.

In [ ]:
# ── Part 5: Custom profiling regions ─────────────────────────────────────────
# Simulate a realistic training step with multiple sub-operations
N_STEPS_PROFILE = 3
step_times = {'tokenize': [], 'to_device': [], 'forward': [], 'loss': [], 'backward': [], 'step': []}

with profile(activities=activities, record_shapes=False) as prof_detailed:
    for step in range(N_STEPS_PROFILE):

        with record_function(f"step_{step}/tokenize"):
            t0 = time.perf_counter()
            # Simulate tokenization (CPU-bound)
            x_cpu = torch.randint(0, VOCAB, (B, S))
            step_times['tokenize'].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/to_device"):
            t0 = time.perf_counter()
            x_gpu = x_cpu.to(DEVICE)
            y_gpu = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
            if HAS_GPU: torch.cuda.synchronize()
            step_times['to_device'].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/forward"):
            t0 = time.perf_counter()
            logits = model(x_gpu)
            if HAS_GPU: torch.cuda.synchronize()
            step_times['forward'].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/loss"):
            t0 = time.perf_counter()
            loss_val = criterion(logits.view(-1, VOCAB), y_gpu.view(-1))
            step_times['loss'].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/backward"):
            t0 = time.perf_counter()
            optimizer.zero_grad(); loss_val.backward()
            if HAS_GPU: torch.cuda.synchronize()
            step_times['backward'].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/optimizer_step"):
            t0 = time.perf_counter()
            optimizer.step()
            if HAS_GPU: torch.cuda.synchronize()
            step_times['step'].append((time.perf_counter() - t0) * 1000)

print("Detailed step timing (median across 3 steps):")
total_recorded = 0
for op, times_list in step_times.items():
    t = np.median(times_list)
    total_recorded += t
    print(f"  {op:20s}: {t:.3f} ms")
print(f"  {'TOTAL':20s}: {total_recorded:.3f} ms")
print()
print("\u2192 record_function regions appear as named spans in the Chrome trace viewer.")
print("  Open the saved .json trace at: chrome://tracing  (or ui.perfetto.dev)")

# Save trace
trace_path = "profiler_trace.json"
prof_detailed.export_chrome_trace(trace_path)
print(f"  Trace saved to: {os.path.abspath(trace_path)}")

#### Reading your Chrome trace in 5 steps

1. Open `chrome://tracing` in Chrome (or [ui.perfetto.dev](https://ui.perfetto.dev) for a modern UI)
2. Click **Load** (top-left button) → select the `profiler_trace.json` file printed above
3. **Rows** = CPU threads; GPU kernels appear as additional rows labeled with CUDA stream names
4. Your `record_function` names appear as **colored spans** — look for `step_N/backward`, `step_N/forward`, etc.
5. Navigate: **W/S** = zoom in/out, **A/D** = pan left/right; click any span for exact duration in the bottom panel

**What to look for first:** The widest span in the backward row — that's your current bottleneck. Gaps between spans mean the GPU was idle waiting for the CPU to dispatch the next operation.

---

## Part 6 — End-to-End: Profiling a Real Training Step

Now we profile a step closer to what the `04-llm` fine-tuning loop does: load a batch from a DataLoader, run forward/backward, and time each phase.

#### 🔮 Predict first

Across a full training step (data loading + forward + backward + optimizer), which component typically surprises practitioners most?

1. **(a) Data loading** — they assume the GPU is always busy; disk/CPU preprocessing can be the bottleneck
2. **(b) The optimizer step** — AdamW with momentum state updates is often neglected
3. **(c) The forward pass** — practitioners often focus on backward, underestimating forward complexity

The answer varies by hardware and batch size — the profiler tells you which is true for YOUR setup.

In [ ]:
# ── Part 6: End-to-end step timing with DataLoader ───────────────────────────
from torch.utils.data import TensorDataset, DataLoader

# Simulate a realistic dataset (in-memory for this demo)
N_SAMPLES = 1000
dummy_x = torch.randint(0, VOCAB, (N_SAMPLES, S))
dummy_y = torch.randint(0, VOCAB, (N_SAMPLES, S))
dataset = TensorDataset(dummy_x, dummy_y)
loader  = DataLoader(dataset, batch_size=B, shuffle=True, num_workers=0, pin_memory=HAS_GPU)

phase_records = {k: [] for k in ['load_batch', 'to_device', 'forward', 'backward', 'optimizer']}

model.train()
loader_iter = iter(loader)
N_MEASURED = min(20, len(loader))

for step in range(N_MEASURED):
    t0 = time.perf_counter()
    x_cpu, y_cpu = next(loader_iter)
    if HAS_GPU: torch.cuda.synchronize()
    phase_records['load_batch'].append((time.perf_counter() - t0) * 1000)

    t1 = time.perf_counter()
    x_dev, y_dev = x_cpu.to(DEVICE), y_cpu.to(DEVICE)
    if HAS_GPU: torch.cuda.synchronize()
    phase_records['to_device'].append((time.perf_counter() - t1) * 1000)

    t2 = time.perf_counter()
    logits = model(x_dev)
    loss_val = criterion(logits.view(-1, VOCAB), y_dev.view(-1))
    if HAS_GPU: torch.cuda.synchronize()
    phase_records['forward'].append((time.perf_counter() - t2) * 1000)

    t3 = time.perf_counter()
    optimizer.zero_grad(); loss_val.backward()
    if HAS_GPU: torch.cuda.synchronize()
    phase_records['backward'].append((time.perf_counter() - t3) * 1000)

    t4 = time.perf_counter()
    optimizer.step()
    if HAS_GPU: torch.cuda.synchronize()
    phase_records['optimizer'].append((time.perf_counter() - t4) * 1000)

# Summary
medians = {k: np.median(v) for k, v in phase_records.items()}
total_ms = sum(medians.values())

print(f"End-to-end training step breakdown (median over {N_MEASURED} steps):")
print()
print(f"{'Phase':20s}  {'Time (ms)':10s}  {'% of total':10s}  {'Cumulative':10s}")
print("-" * 55)
cumulative = 0
bottleneck = max(medians, key=medians.get)
for phase, ms in sorted(medians.items(), key=lambda x: -x[1]):
    pct = ms / total_ms * 100
    cumulative += pct
    flag = " \u2190 BOTTLENECK" if phase == bottleneck else ""
    print(f"  {phase:18s}  {ms:8.2f}ms  {pct:8.1f}%  {cumulative:8.1f}%{flag}")
print(f"\n  Total step time: {total_ms:.2f} ms  ({1000/total_ms:.1f} steps/second)")

# The bottleneck and recommended fix
print()
print(f"Closing Decision: The bottleneck is '{bottleneck}' at {medians[bottleneck]/total_ms*100:.0f}% of wall time.")
if bottleneck == 'backward':
    recommendation = "Use gradient checkpointing (Ch2) or mixed precision (Ch2)"
    saving = 25
elif bottleneck == 'load_batch':
    recommendation = "Use num_workers > 0 in DataLoader to prefetch on CPU"
    saving = 40
elif bottleneck == 'forward':
    recommendation = "Try torch.compile or FlashAttention (Ch4)"
    saving = 20
else:
    recommendation = "Profile with more steps to confirm; consider mixed precision"
    saving = 15
print(f"Recommendation: {recommendation}")
print(f"Expected saving: ~{saving}%")

print()
print(f"--- Closing the 45-second mystery ---")
steps_per_epoch = 1000  # ← substitute len(train_loader) for your actual loader
epoch_s_estimate = (total_ms / 1000) * steps_per_epoch
print(f"  Assuming {steps_per_epoch} steps/epoch:")
print(f"  Estimated epoch time: ~{epoch_s_estimate:.0f}s")
print(f"  The '{bottleneck}' phase owns ~{(medians[bottleneck]/1000 * steps_per_epoch):.0f}s of that.")
print(f"  (Substitute: steps_per_epoch = len(your_train_loader))")

---

## Summary and Closing Decision

| Part | Tool | Key finding |
|------|------|------------|
| 1 | torch.profiler | Identified top operators and phase breakdown |
| 2 | Profiling overhead | Profiler adds 5–15% overhead — use on selected steps only |
| 3 | Compute vs. memory bound | Softmax is memory-bound; matmul is compute-bound |
| 4 | torch.compile | Helps compute-bound ops; diminishing returns on bandwidth-limited workloads |
| 5 | record_function | Named regions visible in Chrome trace (chrome://tracing) |
| 6 | End-to-end timing | Actual bottleneck identified from real training steps |

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- `torch.profiler` — captured CPU+CUDA operator timeline; extracted phase breakdowns
- Profiling overhead — measured two-sided: too much profiling slows training; none = flying blind
- Compute vs. memory bound — softmax vs. matmul compared; memory bottleneck identified
- `torch.compile` — speedup measured; break-even calculated
- `record_function` — custom regions created; Chrome trace saved
- End-to-end step timing — per-phase percentages measured on a real DataLoader

### Tier 2 — Explained but Not Built
- **PyTorch Memory Snapshot** — `torch.cuda.memory._snapshot()` captures full allocation graph; shown but not built

### Tier 3 — Named but Out of Scope
- **Nsight Systems** — NVIDIA's production GPU profiler; shows CUDA kernels, PCIe transfers, and SM utilization at hardware level
- **Perfetto** — Google's open-source trace viewer; compatible with Chrome trace format
- **DCGM** — Data Center GPU Manager; cluster-wide GPU utilization monitoring

---

## When to Use What

| Situation | Tool | Why |
|---|---|---|
| "Something is slow, I don't know what" | `torch.profiler` for 3–5 steps | Full operator timeline |
| "I want to time one function quickly" | `record_function` + `time.perf_counter` | Low overhead |
| "Is my GPU actually busy?" | `torch.cuda.utilization()` | Hardware utilization check |
| "My model uses lots of small ops" | `torch.compile` | Fuses ops; removes Python overhead |
| "Why is attention slow at long sequences?" | FlashAttention (Ch4) | Algorithmic fix: no S×S materialization |

→ **Next:** `learning/ai-infrastructure/04-flash-attention/` — profiling showed that attention softmax is memory-bound at S≥512. The next chapter explains exactly *why* and how FlashAttention solves it without any accuracy change.